# Trabalho Prático Final - Parte 2
## Interpretação do Melhor Modelo

Objetivo: interpretar o melhor modelo treinado usando permutation importance e, opcionalmente, SHAP.

### Configuração Inicial

Nesta célula, importamos as bibliotecas necessárias para a análise, incluindo `pandas` para manipulação de dados, `numpy` para operações numéricas, `matplotlib` e `seaborn` para visualização, `os` para gerenciamento de arquivos, `joblib` para carregar o modelo treinado, e `permutation_importance` e `f1_score` do `sklearn` para avaliação e interpretação do modelo. Também configuramos um `RANDOM_SEED` para reprodutibilidade e criamos os diretórios para salvar os resultados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings

from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs("../results/figures/interpretability", exist_ok=True)
os.makedirs("../results/metrics", exist_ok=True)

### Carregamento dos Dados de Teste

Nesta etapa, carregamos o conjunto de dados de teste (`t2_test.csv`), que será utilizado para avaliar o modelo e calcular a importância das features. Separamos as features (`X_test`) da variável alvo (`y_test`).

In [ ]:
target_col = "Overall"

test_df = pd.read_csv("../data/processed/t2_test.csv")
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(X_test.shape)
print(y_test.shape)

### Recuperação do Nome do Melhor Modelo

Lemos o nome do melhor modelo treinado a partir de um arquivo de texto, que foi salvo durante a etapa de treinamento. Isso garante que estamos interpretando o modelo que obteve o melhor desempenho.

In [ ]:
with open("../results/metrics/best_model_name.txt", "r", encoding="utf-8") as f:
    best_model_name = f.read().strip()

print(f"Melhor modelo: {best_model_name}")

### Carregamento do Melhor Modelo

Utilizamos o `joblib` para carregar o modelo serializado (`.pkl`) com base no nome do melhor modelo obtido na etapa anterior. Este é o modelo que será interpretado.

In [ ]:
model_filename = best_model_name.lower().replace(" ", "_") + "_best_model.pkl"
model_path = f"../results/models/{model_filename}"

best_model = joblib.load(model_path)

print(f"Modelo carregado de: {model_path}")

### Cálculo da Importância por Permutação

Aqui, calculamos a importância de cada feature usando o método de Permutation Importance. Este método avalia a relevância de cada atributo medindo a queda no desempenho do modelo (usando o F1-score) quando os valores desse atributo são embaralhados aleatoriamente. Os resultados são salvos em um DataFrame e exportados para um arquivo CSV.

In [ ]:
perm_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

importance_df = importance_df.sort_values("importance_mean", ascending=False)

importance_df.to_csv("../results/metrics/t2_permutation_importance.csv", index=False)

importance_df.head(20)

### Visualização das Top 20 Features por Permutation Importance

Nesta célula, geramos um gráfico de barras para visualizar as 20 features mais importantes, conforme determinado pela Permutation Importance. O gráfico mostra a queda média no F1-score para cada feature e as barras de erro indicam a variação (desvio padrão) da importância. O gráfico é salvo como uma imagem.

In [ ]:
top_features = importance_df.head(20)

plt.figure(figsize=(10, 8))

# Plotar as barras sem as barras de erro
sns.barplot(
    data=top_features,
    x="importance_mean",
    y="feature"
)

# Adicionar as barras de erro manualmente
# A posição y para as barras é 0, 1, 2, ... para cada feature
y_positions = np.arange(len(top_features))
plt.errorbar(
    x=top_features["importance_mean"],
    y=y_positions,
    xerr=top_features["importance_std"],
    fmt='none', # Não desenhar linha entre os pontos de erro
    c='black',  # Cor das barras de erro
    capsize=4   # Tamanho das "tampas" das barras de erro
)

plt.title(f"Top 20 Features por Permutation Importance - {best_model_name}")
plt.xlabel("Queda média no F1-score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("../results/figures/interpretability/permutation_importance_top20.png", dpi=300)
plt.show()

### Análise de Interpretabilidade com SHAP (Opcional)

Esta seção tenta realizar uma análise de interpretabilidade mais aprofundada usando a biblioteca SHAP (SHapley Additive exPlanations). É feita uma amostragem dos dados de teste para otimizar o custo computacional. Se o SHAP estiver disponível e o modelo for compatível, um `summary_plot` é gerado para visualizar a importância e o impacto de cada feature nas previsões do modelo. Em caso de erro (por exemplo, SHAP não instalado ou modelo incompatível), uma mensagem de fallback é exibida.

In [ ]:
try:
    import shap

    print("SHAP disponível. Gerando análise SHAP...")

    # Amostra para reduzir custo computacional
    X_sample = X_test.sample(
        n=min(500, len(X_test)),
        random_state=RANDOM_SEED
    )

    # Para modelos em pipeline, tentar extrair modelo final e dados transformados
    preprocessed_sample = best_model[:-1].transform(X_sample)
    final_estimator = best_model[-1]

    explainer = shap.TreeExplainer(final_estimator)
    shap_values = explainer.shap_values(preprocessed_sample)

    shap.summary_plot(
        shap_values,
        preprocessed_sample,
        feature_names=X_test.columns,
        show=False
    )

    plt.tight_layout()
    plt.savefig("../results/figures/interpretability/shap_summary_plot.png", dpi=300)
    plt.show()

except Exception as e:
    print("SHAP não foi executado.")
    print(f"Motivo: {e}")
    print("A análise principal de interpretabilidade será baseada em permutation importance.")

## Texto-base para o relatório

A interpretação do modelo final foi realizada por meio de permutation importance. Esse método avalia a relevância de cada atributo medindo a queda no desempenho do modelo quando os valores desse atributo são embaralhados. Assim, atributos cuja permutação reduz mais fortemente o F1-score são considerados mais importantes para a decisão do modelo.

Foram analisadas as 20 features mais relevantes. Os resultados permitem identificar quais descritores moleculares tiveram maior impacto na predição de mutagenicidade Ames. Essa análise oferece uma visão global do comportamento do modelo, embora não estabeleça causalidade entre os descritores e a mutagenicidade.